# Downloading Tide Data from NOAA CO-OPS API

Author of this document: Timothy Divoll

The purpose of this notebook is to demonstrate how to download data from NOAA's CO-OPS Data API. In this example, data are parsed into a dataframe and also written to CSV files (as needed to use in the `Assessing Accuracy of the Tide Predictions of the Ocean State Ocean Model` notebook.

If needed, dataframes could be saved in other common formats and exported as needed.

First, we need to install the noaa_coops Python wrapper (https://github.com/GClunies/noaa_coops)

In [55]:
%pip install noaa_coops -q # this package sends requests to the NOAA CO-OPS Data API

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621

[notice] A new release of pip available: 22.2.2 -> 22.3.1
[notice] To update, run: /opt/homebrew/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import dependencies and ignore warnings in code outputs
import noaa_coops
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

### Direct data dowload - example
The next cell has code to extract the data directly from the NOAA CO-OPS API rather than downloading from the webpage. This block only shows two example stations and additional stations could be added as needed.

In [71]:
# send a request to the CO-OPS API for Data Retrieval
# https://api.tidesandcurrents.noaa.gov/api/prod/
# MLLW = mean lower low water

station_list = [8461490, 8510560, 8447930, 8441930, 8452660, 8454049, 8447386, 8452944, 8454000]

# New London, CT example
new_london = noaa_coops.Station(station_list[0]) #use a different index for a different station
new_london_verified = new_london.get_data(
    begin_date = "20040101",
    end_date = "20040616",
    product = "water_level",
    datum = "MLLW",
    units = "metric",
    time_zone = "gmt",
    interval = "h")
new_london_predicted = new_london.get_data(
    begin_date = "20040101",
    end_date = "20040616",
    product = "predictions",
    datum = "MLLW",
    units = "metric",
    time_zone = "gmt",
    interval = "h")

# merge verified and predicted, then rename columns to match `readcsv` function
new_london_df = pd.merge(new_london_verified, new_london_predicted, on="date_time").drop(columns = ["sigma", "flags", "QC"]).rename(columns={"water_level": "Verified (m)", "predicted_wl": "Predicted (m)"}).reset_index()
new_london_df["Date"] = new_london_df["date_time"].dt.strftime("%m/%d/%Y")
new_london_df["Time (GMT)"] = new_london_df["date_time"].dt.strftime("%H:%M")


# Montauk, NY example
montauk = noaa_coops.Station(station_list[1])
montauk_verified = montauk.get_data(
    begin_date = "20040101",
    end_date = "20220616",
    product = "water_level",
    datum = "MLLW",
    units = "metric",
    time_zone = "gmt",
    interval = "h")
montauk_predicted = montauk.get_data(
    begin_date = "20040101",
    end_date = "20220616",
    product = "predictions",
    datum = "MLLW",
    units = "metric",
    time_zone = "gmt",
    interval = "h")

# merge verified and predicted, then rename columns to match `readcsv` function in `Assessing Accuracy of the Tide Predictions of the Ocean State Ocean Model` notebook
montauk_df = pd.merge(montauk_verified, montauk_predicted, on="date_time").drop(columns = ["sigma", "flags", "QC"]).rename(columns={"water_level": "Verified (m)", "predicted_wl": "Predicted (m)"}).reset_index()
montauk_df["Date"] = montauk_df["date_time"].dt.strftime("%m/%d/%Y")
montauk_df["Time (GMT)"] = montauk_df["date_time"].dt.strftime("%H:%M")

View the dataframes for each station.

In [72]:
new_london_df.head()

,date_time,Verified (m),Predicted (m),Date,Time (GMT)
0,2004-01-01 00:00:00,0.353,0.408,01/01/2004,00:00
1,2004-01-01 01:00:00,0.238,0.311,01/01/2004,01:00
2,2004-01-01 02:00:00,0.119,0.194,01/01/2004,02:00
3,2004-01-01 03:00:00,0.101,0.104,01/01/2004,03:00
4,2004-01-01 04:00:00,0.118,0.098,01/01/2004,04:00


In [73]:
montauk_df.tail()

,date_time,Verified (m),Predicted (m),Date,Time (GMT)
167011,2022-06-16 19:00:00,0.403,0.256,06/16/2022,19:00
167012,2022-06-16 20:00:00,0.273,0.132,06/16/2022,20:00
167013,2022-06-16 21:00:00,0.156,0.036,06/16/2022,21:00
167014,2022-06-16 22:00:00,0.141,0.035,06/16/2022,22:00
167015,2022-06-16 23:00:00,0.266,0.163,06/16/2022,23:00


Export each file to a CSV in the format expected by `readcsv` in the `Assessing Accuracy of the Tide Predictions of the Ocean State Ocean Model` notebook.

In [74]:
new_london_df.to_csv("/Users/tdivoll/Downloads/new_london_tide_data.csv")

In [75]:
montauk_df.to_csv("/Users/tdivoll/Downloads/montauk_tide_data.csv")